# Train YOLO with Goalpost Detection (SoccerNet-v3)

This notebook fixes the missing **goalpost** class. SoccerNet stores goal parts as **line** annotations; the converter below turns them into YOLO boxes.

## Kaggle setup (do this first)

1. **New notebook** on Kaggle
2. **Settings** → Accelerator: **GPU T4 x2** (or P100)
3. **Settings** → Internet: **On**
4. **Add Input** → **Models** → add your current `best.pt` from `~/Desktop/statsapp/models/`
5. Upload this notebook (`training/kaggle_goalpost_train.ipynb`) or paste cells
6. **Run All** (takes ~1–2 hours)

## After training (on your Mac)

```bash
# Download best.pt from Kaggle Output tab, then:
cp ~/Downloads/best.pt ~/Desktop/statsapp/models/best.pt
cd ~/Desktop/statsapp
python yolo_inference.py    # should show goalpost boxes on video
python main.py              # full stats pipeline
```

**Success check:** Cell 4 must print thousands of goalpost labels. Cell 6 validation must show `goalpost` mAP50 > 0.

In [ ]:
!pip install -q ultralytics SoccerNet tqdm

In [ ]:
from pathlib import Path

from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames

SOCCERNET_DIR = Path("/kaggle/working/SoccerNet")
YOLO_DIR = Path("/kaggle/working/soccernet_yolo")
MAX_GAMES = 20  # increase to 50+ if you have disk; Kaggle limit ~19.5 GiB

downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))
games = getListGames("train", task="frames")[:MAX_GAMES]
print(f"Downloading {len(games)} games...")

for i, game in enumerate(games, 1):
    print(f"[{i}/{len(games)}] {game}")
    downloader.downloadGame(
        game=game,
        files=["Frames-v3.zip", "Labels-v3.json"],
        spl="train",
    )

!df -h /kaggle/working
print("Download complete")

In [ ]:
# Write the fixed converter (goal lines -> YOLO goalpost boxes)
from pathlib import Path

CONVERTER = Path("/kaggle/working/soccernet_to_yolo.py")
CONVERTER.write_text(Path("soccernet_to_yolo.py").read_text() if Path("soccernet_to_yolo.py").exists() else r'''
"""Convert SoccerNet-v3 JSON to Ultralytics YOLO format."""
from __future__ import annotations
import json, shutil, zipfile
from pathlib import Path
from tqdm import tqdm

YOLO_NAMES = ["ball", "player", "goalkeeper", "referee", "goalpost"]
SN_BBOX_CLASS_TO_YOLO = {
    "Ball": 0, "Player team left": 1, "Player team right": 1,
    "Player team unknown 1": 1, "Player team unknown 2": 1,
    "Goalkeeper team left": 2, "Goalkeeper team right": 2, "Goalkeeper team unknown": 2,
    "Main referee": 3, "Side referee": 3,
}
SN_GOAL_LINE_CLASS_TO_YOLO = {
    "Goal left post left ": 4, "Goal left post right": 4, "Goal left crossbar": 4,
    "Goal right post left": 4, "Goal right post right": 4, "Goal right crossbar": 4,
}

def _xyxy_to_yolo_line(cls_id, x1, y1, x2, y2, image_meta):
    w_img, h_img = float(image_meta["width"]), float(image_meta["height"])
    x_c = min(1.0, max(0.0, ((x1 + x2) / 2.0) / w_img))
    y_c = min(1.0, max(0.0, ((y1 + y2) / 2.0) / h_img))
    bw = min(1.0, max(0.0, abs(x2 - x1) / w_img))
    bh = min(1.0, max(0.0, abs(y2 - y1) / h_img))
    if bw <= 0 or bh <= 0: return None
    return f"{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}"

def bbox_to_yolo_line(bbox, image_meta):
    sn_class = bbox.get("class")
    if sn_class not in SN_BBOX_CLASS_TO_YOLO: return None
    p = bbox["points"]
    return _xyxy_to_yolo_line(SN_BBOX_CLASS_TO_YOLO[sn_class], float(p["x1"]), float(p["y1"]), float(p["x2"]), float(p["y2"]), image_meta)

def line_to_yolo_line(line, image_meta, padding=12.0):
    sn_class = line.get("class")
    if sn_class not in SN_GOAL_LINE_CLASS_TO_YOLO: return None
    points = line.get("points") or []
    if len(points) < 4: return None
    xs = [float(points[i]) for i in range(0, len(points), 2)]
    ys = [float(points[i]) for i in range(1, len(points), 2)]
    return _xyxy_to_yolo_line(SN_GOAL_LINE_CLASS_TO_YOLO[sn_class], min(xs)-padding, min(ys)-padding, max(xs)+padding, max(ys)+padding, image_meta)

def _extract_image(zip_path, image_name, dest_path):
    if dest_path.exists(): return True
    if not zip_path.exists(): return False
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        if image_name not in zf.namelist(): return False
        with zf.open(image_name) as src, open(dest_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return True

def convert_game(soccernet_root, game_rel_path, images_out, labels_out, stem_prefix):
    labels_path = soccernet_root / game_rel_path / "Labels-v3.json"
    if not labels_path.exists(): return 0
    metadata = json.loads(labels_path.read_text(encoding="utf-8"))
    zip_path = soccernet_root / metadata["GameMetadata"]["UrlLocal"] / "Frames-v3.zip"
    count = 0
    for action_name in metadata["GameMetadata"]["list_actions"]:
        img_names = [action_name] + metadata["actions"][action_name]["linked_replays"]
        for i, img_name in enumerate(img_names):
            img_type = "actions" if i == 0 else "replays"
            ann = metadata[img_type][img_name]
            yolo_lines = [l for b in ann.get("bboxes", []) if (l := bbox_to_yolo_line(b, ann["imageMetadata"]))]
            yolo_lines += [l for g in ann.get("lines", []) if (l := line_to_yolo_line(g, ann["imageMetadata"]))]
            if not yolo_lines: continue
            safe_stem = f"{stem_prefix}_{img_name.replace('/', '_').replace('.png', '')}"
            image_out = images_out / f"{safe_stem}.png"
            label_out = labels_out / f"{safe_stem}.txt"
            if not _extract_image(zip_path, img_name, image_out): continue
            label_out.write_text("\n".join(yolo_lines) + "\n", encoding="utf-8")
            count += 1
    return count

def write_data_yaml(output_dir):
    output_dir = Path(output_dir)
    val = output_dir / "images" / "val"
    val_path = "images/val" if val.exists() and any(val.glob("*")) else "images/train"
    test_path = "images/test" if (output_dir / "images" / "test").exists() else val_path
    yaml_path = output_dir / "data.yaml"
    yaml_path.write_text(
        f"path: {output_dir.resolve()}\ntrain: images/train\nval: {val_path}\ntest: {test_path}\n"
        f"nc: {len(YOLO_NAMES)}\nnames: {YOLO_NAMES}\n",
        encoding="utf-8",
    )
    return yaml_path

def convert_soccernet_v3(soccernet_root, output_dir, splits=None, max_games_per_split=None):
    from SoccerNet.utils import getListGames
    soccernet_root, output_dir = Path(soccernet_root), Path(output_dir)
    splits = splits or ["train"]
    split_map = {"train": "train", "valid": "val", "test": "test"}
    total = 0
    for split in splits:
        yolo_split = split_map.get(split, split)
        images_out = output_dir / "images" / yolo_split
        labels_out = output_dir / "labels" / yolo_split
        images_out.mkdir(parents=True, exist_ok=True)
        labels_out.mkdir(parents=True, exist_ok=True)
        games = getListGames(split, task="frames")
        if max_games_per_split is not None: games = games[:max_games_per_split]
        for game in tqdm(games, desc=f"Converting {split}"):
            prefix = game.replace("/", "_").replace(" ", "_")
            total += convert_game(soccernet_root, game, images_out, labels_out, prefix)
    yaml_path = write_data_yaml(output_dir)
    print(f"Converted {total} images -> {output_dir}")
    return yaml_path
'''.strip() + "\n", encoding="utf-8")

print("Wrote", CONVERTER)

In [ ]:
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from soccernet_to_yolo import convert_soccernet_v3

yaml_path = convert_soccernet_v3(
    soccernet_root=SOCCERNET_DIR,
    output_dir=YOLO_DIR,
    splits=["train"],
    max_games_per_split=MAX_GAMES,
)

# Free disk after images extracted
for zip_path in Path(SOCCERNET_DIR).rglob("Frames-v3.zip"):
    zip_path.unlink()
print("Deleted Frames-v3.zip to save disk")

# --- CRITICAL: verify goalpost labels exist ---
class_counts = Counter()
for label_file in (YOLO_DIR / "labels" / "train").glob("*.txt"):
    for line in label_file.read_text().splitlines():
        if line.strip():
            class_counts[int(line.split()[0])] += 1

names = ["ball", "player", "goalkeeper", "referee", "goalpost"]
print("\nLabel counts per class:")
for i, name in enumerate(names):
    print(f"  {name}: {class_counts.get(i, 0)}")

goalpost_count = class_counts.get(4, 0)
if goalpost_count == 0:
    raise RuntimeError(
        "ZERO goalpost labels! Converter bug — do not train. "
        "Goal lines must be converted from ann['lines']."
    )
print(f"\nOK: {goalpost_count} goalpost labels found. Safe to train.")
print("data.yaml:", yaml_path)
!df -h /kaggle/working

In [ ]:
import shutil
from pathlib import Path
from ultralytics import YOLO

WEIGHTS_IN = next(Path("/kaggle/input").rglob("best.pt"), None)
if WEIGHTS_IN:
    shutil.copy(WEIGHTS_IN, "/kaggle/working/best.pt")
    MODEL = "/kaggle/working/best.pt"
    print("Fine-tuning from:", WEIGHTS_IN)
else:
    MODEL = "yolov9c.pt"
    print("No best.pt in /kaggle/input — training from scratch:", MODEL)
    print("Tip: Add your models/best.pt as a Kaggle Model input for faster/better results")

model = YOLO(MODEL)
results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project="/kaggle/working/runs",
    name="soccernet_goalpost",
    device=0,
    lr0=1e-4,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.1,
)
print("Training done:", results.save_dir / "weights" / "best.pt")

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
from ultralytics import YOLO

best = next(Path("/kaggle/working/runs").rglob("best.pt"), None)
if best is None:
    raise FileNotFoundError("No best.pt under /kaggle/working/runs")

print("Best weights:", best)
metrics = YOLO(str(best)).val(data=str(yaml_path))
print("\nPer-class mAP50 (look for goalpost > 0):")
if hasattr(metrics, "results_dict"):
    for k, v in sorted(metrics.results_dict.items()):
        if "goalpost" in k.lower() or "mAP50" in k:
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

display(FileLink(str(best)))
print("\nDownload best.pt from the link above, then on your Mac:")
print("  cp ~/Downloads/best.pt ~/Desktop/statsapp/models/best.pt")
print("  cd ~/Desktop/statsapp && python yolo_inference.py")